# Implementation of Lambert Liu

### Imports and config dicts

In [2]:
import numpy as np 
import polars as pl 
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [3]:
config_dict = {
    # Binning configs
    'fine_bin_mins' : 5,
    'coarse_bin_mins' : 60,

    # Train, burn in, test validation split config
    'train_days' : 7,
    'burn_in_days' : 7,
    'validation_days' : 7,
    'test_days' : 7,
    
    # Min Params
    'mean_min' : 1e-8,
    'var_min' : 1e-8,
    'min_mean_var_diff' : 1e-8,
    
    # Smoothing Strength and w
    'smoothing_strength' : 0.2, 
    'w' : 0.1}

### Loading and saving data 

In [4]:
data_path = '/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate'
input_data = 'train_df'
df = pl.scan_parquet(f'{data_path}/input/{input_data}.parquet')


# Creating functions for storing and reading data
def store_data(data, filename, data_path=data_path):
    if isinstance(data, pl.LazyFrame):
        data.sink_parquet(f'{data_path}/output/{filename}.parquet')
    elif isinstance(data, np.ndarray):
        np.save(f'{data_path}/output/{filename}.npy', data)
    elif isinstance(data, pl.DataFrame):
        data.write_parquet(f'{data_path}/output/{filename}.parquet')
    else:
        raise TypeError('Function doesnt support this data type')
    

def load_data(filename, data_type, data_path=data_path):
    ''' 
    Arguments:
        filename: 
            the saved file name
        data_type: ['lazy', 'np', 'df']
            the type of data we want to load in
    '''
    if data_type == 'lazy':
        data = pl.scan_parquet(f'{data_path}/{filename}.parquet')
    elif data_type == 'np':
        data = np.load(f'{data_path}/{filename}.npy')
    elif data_type == 'df':
        data = pl.read_parquet(f'{data_path}/{filename}.parquet')
    else:
        raise TypeError('Function doesnt support this data type')
    return data

### Getting helper dictionaries

In [5]:
# Helper functions outputing dicitonaries
def get_bin_metrics(config_dict=config_dict):
    ''' 
    Returns a dictionary containing all information needed about bin length these numbers are needed throughout the pipeline
    '''
    # Check that bin length divides number of days
    assert 24 * 60 % config_dict['fine_bin_mins'] == 0
    assert 24 * 60 % config_dict['coarse_bin_mins'] == 0
    assert config_dict['coarse_bin_mins'] % config_dict['fine_bin_mins'] == 0

    output_dict = {'fine_bins_per_day' : 24 * 60 // config_dict['fine_bin_mins']}
    output_dict['fine_bins_per_week'] = output_dict['fine_bins_per_day'] * 7
    output_dict['fine_bins_per_coarse_bin'] = config_dict['coarse_bin_mins'] // config_dict['fine_bin_mins']
    output_dict['coarse_bins_per_week'] = output_dict['fine_bins_per_week'] // output_dict['fine_bins_per_coarse_bin']
    output_dict['fine_bin_seconds'] = config_dict['fine_bin_mins'] * 60 

    return output_dict

def get_train_test_split(config_dict=config_dict):
    ''' 
    Calculates the fine bin index of the train test and validation period start and end
    Returns a dictionary with all this information in
    '''

    fine_bins_per_day = 24 * 60 // config_dict['fine_bin_mins']

    train_bins = fine_bins_per_day * config_dict['train_days']
    burn_in_bins = fine_bins_per_day * config_dict['burn_in_days']
    validation_bins = fine_bins_per_day * config_dict['validation_days']
    test_bins = fine_bins_per_day * config_dict['test_days']

    # Constructing a dict for the output
    output_dict = {'train_start' : 0,
                   'train_end' : train_bins}
    
    output_dict['burn_in_start'] = output_dict['train_end']
    output_dict['burn_in_end'] = output_dict['burn_in_start'] + burn_in_bins

    output_dict['validation_start'] = output_dict['burn_in_end']
    output_dict['validation_end'] = output_dict['validation_start'] + validation_bins

    output_dict['test_start'] = output_dict['validation_end']
    output_dict['test_end'] = output_dict['test_start'] + test_bins
    
    return output_dict
    
# Adding the fine_bins_per_coarse_bin to the train_test_split_dict
def add_training_denom(bin_metric_dict, config_dict=config_dict):
    ''' 
    Finds how many fine bins are used for the estimate of the iniatial parameter.
    This is the denominator on the mean estimate as it is counts/bins and is similarly used in the variance estimate.

    This fucntion assumes train days is a multiple of 7 to work

    Returns:
        An updated bin metric dict with a new column train denom
    '''
    assert config_dict['train_days'] % 7 == 0
    bin_metric_dict['train_denom'] = bin_metric_dict['fine_bins_per_coarse_bin'] * (config_dict['train_days'] // 7)
    return bin_metric_dict

In [6]:
# Uses the above functions to get 2 dictionaries 
# train_test_dict with the fine bins index upon which we make the train test split
# bin metric dict which calculates metric such as fine bins per day and fine bins used in the training estimate
train_test_dict = get_train_test_split()
bin_metric_dict = get_bin_metrics()
bin_metric_dict = add_training_denom(bin_metric_dict)

### Preprocessing data

This involves:

    - Turning the source_user@domain column into a user_id to make it more lightweight
    - Making the fine bin column and getting counts in each fine bin
    - Making the course bin column
    - Getting the fine bin within coarse bin position

In [7]:
# Creating a fine bin and identifying users with some counts in the bin

def create_counts_data(df):
    ''' 
    Gets the counts per fine bin x source user from the data
    '''
    df = df.with_columns(time = pl.col('time').dt.total_seconds())
    df = df.with_columns(fine_bin_id = pl.col('time') // bin_metric_dict['fine_bin_seconds'])
    user_x_fine_bin_cnts = df.group_by(['source_user@domain', 'fine_bin_id']).agg(pl.len().alias('count')).collect()
    return user_x_fine_bin_cnts

def create_user_to_id_mapping(users_df, mapping_file_name):
        ''' 
        Creates a table containing the 
        '''
        # Creating a user lookup table and storing it
        user_mapping = users_df.select('source_user@domain').unique().sort(by='source_user@domain').with_row_index('user_id')
        store_data(user_mapping, mapping_file_name)
        
        # Joining on the lookup table and dropping columns
        users_df = users_df.join(user_mapping, on='source_user@domain', how='inner')
        users_df = users_df.select(['user_id', 'fine_bin_id', 'count']).sort(['user_id', 'fine_bin_id'])

        return users_df, user_mapping

def create_coarse_bins(users_df):
        ''' 
        takes a DF and creates two new columns:
            coarse_bin_id
            fine_bin_within_coarse_pos
        '''
        # Creating columns needed fr
        users_df = users_df.with_columns(fine_bin_pos_in_week = pl.col('fine_bin_id') % bin_metric_dict['fine_bins_per_week'])
        users_df = users_df.with_columns(
            coarse_bin_id = pl.col('fine_bin_pos_in_week') // bin_metric_dict['fine_bins_per_coarse_bin'],
            fine_bin_within_coarse_pos = pl.col('fine_bin_pos_in_week') % bin_metric_dict['fine_bins_per_coarse_bin'])

        return users_df

In [8]:
# Doing initial manipulations on the data including replacing source_user@domain with user_id and constructing course and fine bins
df_counts = create_counts_data(df)
df_counts, user_mapping = create_user_to_id_mapping(df_counts, mapping_file_name='source_users_to_id_mapping')
df_counts = create_coarse_bins(df_counts)

### Getting inital parameter estimates and interpolation weights

In [9]:
def get_training_sums(df_counts, train_test_dict=train_test_dict):
    ''' 
    Creates the training data and the sum of counts and sum of counts squared columns

    This data is used to get the initial parameter estimates
    '''

    train_df = df_counts.filter((pl.col('fine_bin_id') >= train_test_dict['train_start'])
                                    & (pl.col('fine_bin_id') < train_test_dict['train_end']))

    train_df = train_df.with_columns(count_2 = pl.col('count') ** 2)
    train_df = train_df.group_by(['user_id', 'coarse_bin_id']).agg(pl.sum('count').alias('sum_cnt'), pl.sum('count_2').alias('sum_cnt_2'))
    
    return train_df 


def create_init_grids(df_counts, n_users, n_coarse_bins, bin_metric_dict=bin_metric_dict):
    ''' 
    Creates the u and v init grids from the training data
    '''
    # Getting the sum of counts and sum of counts squared needed for mean and variance calculations
    train_df = get_training_sums(df_counts)
    
    # Init a grid of parmeters to use
    u_init = np.zeros((n_users, n_coarse_bins))
    v_init = np.zeros((n_users, n_coarse_bins))

    # Extrating the entries to assign and assigning them to the df
    entries_to_assign = train_df.select(['user_id', 'coarse_bin_id']).to_numpy()

    u_init[entries_to_assign[:,0], entries_to_assign[:,1]] = train_df['sum_cnt'].to_numpy() / bin_metric_dict['train_denom']
    v_init[entries_to_assign[:,0], entries_to_assign[:,1]] = (train_df['sum_cnt_2'].to_numpy() - 
                                                            ((train_df['sum_cnt'] **2) / bin_metric_dict['train_denom']))/(bin_metric_dict['train_denom'] - 1)

    # Capping the min values of u_init and v_init
    u_init = np.maximum(u_init, config_dict['mean_min'])
    v_init = np.maximum(v_init, config_dict['var_min'])

    return u_init, v_init

In [10]:
# Creatung the inital grids and saving the outputs
u_init, v_init = create_init_grids(df_counts, n_users = user_mapping.shape[0], n_coarse_bins = bin_metric_dict['coarse_bins_per_week'])

store_data(u_init, 'u_init')
store_data(v_init, 'v_init')

### Getting interpolation weights

In [11]:
def get_interpolation_weights(bin_metric_dict=bin_metric_dict):
    '''
    A function that returns the weights we apply when interpolating.
    To understand the computation steps see pages 11 and 12 of lambert liu
    returns:
        weights a numpy array which will be applied as w-1 U-1 + w0 U0 + w1 U1
        the weights array has a row for every fine bin in the coarse bin and 3 columns where each column is the weight being applies to Uis
    '''

    # Getting the m (fine bin number, q and r (defined in lambert liu))
    M = bin_metric_dict['fine_bins_per_coarse_bin']
    m = np.arange(1, M+1)
    q = (m - 1)/M
    r = m/M

    # Getting the two terms used in all calculations
    term_1 = r**2 + r*q + q**2
    term_2 = r + q

    # Computing the weights using the lambert and liu formula
    # these come from rearranging the fomula at the top of page 12 for U-1 U0 and U1
    weights = np.zeros((M, 3))

    weights[:, 0] = term_1/6 - term_2/2 + 1/3
    weights[:, 1] = -term_1/3 + term_2/2 + 5/6
    weights[:, 2] = term_1/ 6 - 1/6

    return weights

In [12]:
# Saving the outputs of this section
interpolation_weights = get_interpolation_weights()
store_data(interpolation_weights, filename='interpolation_weights')

### Turning tables to np arrays for use in the numba runner

In [13]:
def counts_df_to_np(df_counts):
    ''' 
    Converts the df_counts into numpy arrays for use in the lambert and liu runner
    '''
    # Converting columns to numpy for use by the numba runner
    cnts_tbl_usr_id = df_counts["user_id"].to_numpy().astype(np.int64)
    cnts_tbl_f_bn_id = df_counts["fine_bin_id"].to_numpy().astype(np.int64)
    cnts_tbl_cnt = df_counts["count"].to_numpy().astype(np.int64)
    return cnts_tbl_usr_id, cnts_tbl_f_bn_id, cnts_tbl_cnt

def create_first_last_interaction_arrays(df_counts):
    '''
    Creates an output table with user first and last interaction indicies within df counts
    Note this is not a fine bin index but a row index
    '''
    # Creating a lookup table for numba for the first and last entry of users 
    user_interactions = df_counts.with_row_index().group_by('user_id').agg(
        user_first_index = pl.min('index'),
        user_last_index = pl.max('index')).sort(by='user_id')

    intract_tbl_usr_id = user_interactions['user_id'].to_numpy().astype(np.int64)
    intract_tbl_frst_int = user_interactions['user_first_index'].to_numpy().astype(np.int64)
    intract_tbl_lst_int = user_interactions['user_last_index'].to_numpy().astype(np.int64)

    return intract_tbl_frst_int, intract_tbl_lst_int, intract_tbl_usr_id

In [14]:
# Running functions and saving arrays
cnts_tbl_usr_id, cnts_tbl_f_bn_id, cnts_tbl_cnt = counts_df_to_np(df_counts)
store_data(cnts_tbl_usr_id, 'cnts_tbl_usr_id')
store_data(cnts_tbl_f_bn_id, 'cnts_tbl_f_bn_id')
store_data(cnts_tbl_cnt, 'cnts_tbl_cnt')

intract_tbl_frst_int, intract_tbl_lst_int, intract_tbl_usr_id = create_first_last_interaction_arrays(df_counts)
store_data(intract_tbl_usr_id, 'intract_tbl_usr_id')
store_data(intract_tbl_frst_int, 'intract_tbl_frst_int')
store_data(intract_tbl_lst_int, 'intract_tbl_lst_int')

### Getting raw and clustered model

In [15]:
def make_raw_model():
    '''
    Creates a dictionary with the model config to be used by the numba runner
    '''
    n_users, n_coarse_bins = u_init.shape
    # Creating a model dict with dummy placeholders for the cluster mean as no smoothing is done for this model
    # There is 1 cluster so cluster mean vectors have first dim 1
    output = {'name' : 'raw_model',
              'cluster_mean_u' : np.zeros((1,n_coarse_bins), dtype='float64'),
              'cluster_mean_v' : np.zeros((1,n_coarse_bins), dtype='float64'),
              'smoothing_parameter' : 0,
              'cluster_assignments' : np.zeros(n_users, dtype='int64')
              }
    
    return output

raw_model = make_raw_model()

In [16]:
### Creating clusters and then initialising a cluster model
#! NOTE right now this clustering doesnt work very well assigning most users to cluster 0 and very few users to the other clusters. 
#! We will work on clustering down the line and check if this will be resolved when we filter out the 0 users

#! NOTE also we should probably cluster the data based on the parameters for the full training data not just for the first week of train

def get_k_means_assignments(k, random_state, vec_to_cluster=u_init):
    '''
    Performs k means clustering on a numpy array and returns a vector of cluster assignments
    '''

    k_means_cluster = KMeans(n_clusters=k, random_state=random_state)
    clusters = k_means_cluster.fit_predict(vec_to_cluster).astype(np.int64)

    return clusters

def get_clustering_means(cluster_groups, u_init=u_init, v_init=v_init):
    ''' 
    Calculates mean u and v values for each cluster group and each time bin
    Used within the get clustering model function
    Args:
        cluster_groups a n_users length vector of cluster assignments
        u_init : the calculated vector of parameter means
        v_init : the calculated vector of initial parameter variances
    '''

    n_users, n_coarse_bins = u_init.shape 
    n_clusters = cluster_groups.max() + 1
    print(f'Number of clusters identified : {n_clusters}')

    # Init mean vectors
    cluster_mean_u = np.zeros((n_clusters, n_coarse_bins), dtype='float64')
    cluster_mean_v = np.zeros((n_clusters, n_coarse_bins), dtype='float64')
    users_per_cluster = np.zeros(n_clusters, dtype='float64')

    # Summing u and v contributions in each cluster
    # Extract the cluster assignment for each user and then add their parameters to each bin
    for user_id in range(n_users):
        cluster_assignment = int(cluster_groups[user_id])
        cluster_mean_u[cluster_assignment, :] += u_init[user_id, :]
        cluster_mean_v[cluster_assignment, :] += v_init[user_id, :]
        users_per_cluster[cluster_assignment] += 1

    # Dividing through to get the averages in each cluster
    for cluster_assignment in range(n_clusters):
        if users_per_cluster[cluster_assignment] > 0:
            cluster_mean_u[cluster_assignment, :] /= users_per_cluster[cluster_assignment]
            cluster_mean_v[cluster_assignment, :] /= users_per_cluster[cluster_assignment]

    return cluster_mean_u, cluster_mean_v


def make_cluster_model(algo_name, cluster_assignments, smoothing_parameter, u_init=u_init, v_init=v_init):
    ''' 
    Creates the clustering model dictionary 
    '''
    cluster_mean_u, cluster_mean_v = get_clustering_means(u_init=u_init, v_init=v_init, cluster_groups=cluster_assignments)

    output = {'name' : f'{algo_name}_clustering_model',
              'cluster_mean_u' : cluster_mean_u,
              'cluster_mean_v' : cluster_mean_v,
              'smoothing_parameter' : smoothing_parameter,
              'cluster_assignments' : cluster_assignments}

    return output 

# running these functions
k_means_assignments = get_k_means_assignments(k=4, random_state=10)
clustering_model = make_cluster_model(algo_name='k_means', 
                                    cluster_assignments=k_means_assignments,
                                    smoothing_parameter=0.5)



Number of clusters identified : 4


### Creating helper functions for the final runner

In [17]:
# Creating functions that get the log pmf value for the negative binomial distribution (or the poisson distirbution for underdispersed users)
from numba import njit
import math 

@njit 
def poisson_lpmf(x, mu):
    ''' 
    Poisson log pmf
    '''
    return x*math.log(mu) - mu - math.lgamma(x+1)

@njit
def neg_bin_lpmf(x, mu, sigma2):
    p = mu/sigma2
    r = (mu*p) / (1-p)
    return math.lgamma(x+r) - math.lgamma(r) - math.lgamma(x+1) + r*math.log(p) + x*math.log(1-p)

@njit 
def get_lmpf_val(x, mu, sigma2, mean_min, var_min, min_mean_var_diff):
    if mu <= 0:
        raise ValueError(' Mu < 0 ')
    if sigma2 <= 0:
        raise ValueError('Sigma^2 , 0')
    
    mu = max(mu, mean_min)
    sigma2 = max(sigma2, var_min)
    if sigma2 <= mu + min_mean_var_diff:
        return poisson_lpmf(x, mu)
    else: 
        return neg_bin_lpmf(x, mu, sigma2)


In [18]:
@njit
def get_time_period(fine_bin_id, validation_start, validation_end, test_start, test_end):
    ''' 
    Returns:
        0 if we are in the validation data
        1 if we are in the test data
        -1 otherwise (train + burn in and any unused data)
    '''
    if validation_start <= fine_bin_id and fine_bin_id < validation_end:
        return 0
    elif test_start <= fine_bin_id and fine_bin_id < test_end:
        return 1
    else: 
        return -1

In [19]:
# Creating a function that smooths between users and parameter
@njit 
def smoothing_function(smoothing_strength, user_parameter, cluster_parameter):
    ''' 
    Function used for smoothing between the cluster parameter and the user parameter
    '''
    return (1-smoothing_strength) * user_parameter + smoothing_strength*cluster_parameter

@njit 
def interpolate_values(v_neg_1, v_0, v_1, fine_bin_within_coarse_pos, interpolation_weights):
    '''
        Applies the quadratic interpolation between the left middle and right bin values
    '''
    # Extract the weights we will use for the interpolation
    w_neg_1, w_0, w_1 = interpolation_weights[fine_bin_within_coarse_pos]
    return w_neg_1 * v_neg_1 + w_0 * v_0 + w_1 * v_1

@njit 
def smooth_params(user_param_grid, cluster_param_grid, cluster_assignments, smoothing_strength, crnt_user, crnt_coarse_bin,
                  crnt_fine_bin_within_coarse_pos, interpolation_weights):
    ''' 
    Takes a parameter mu or sigma2 and smooths it towards the cluster parameter using `smoothing_function`
    '''
    # Get the cluster assignment for the current user
    cluster_id = cluster_assignments[crnt_user]

    # Get the 3 values to interpolate
    n_coarse_bins = user_param_grid.shape[1]
    neg_1_coarse_bin = (crnt_coarse_bin -1) % n_coarse_bins
    _1_coarse_bin = (crnt_coarse_bin + 1)% n_coarse_bins

    ## Smooth the 3 values we will later interpolate
    v_neg_1 = smoothing_function(smoothing_strength, user_param_grid[crnt_user, neg_1_coarse_bin], cluster_param_grid[cluster_id, neg_1_coarse_bin])
    v_0 = smoothing_function(smoothing_strength, user_param_grid[crnt_user, crnt_coarse_bin], cluster_param_grid[cluster_id, crnt_coarse_bin])
    v_1 = smoothing_function(smoothing_strength, user_param_grid[crnt_user, _1_coarse_bin], cluster_param_grid[cluster_id, _1_coarse_bin])

    # Interpolate the values 
    return interpolate_values(v_neg_1, v_0, v_1, crnt_fine_bin_within_coarse_pos, interpolation_weights)



@njit
def get_smoothed_params(u, v, cluster_u, cluster_v, cluster_assignments, smoothing_strength, crnt_user, crnt_coarse_bin,
                        crnt_fine_bin_within_coarse_pos, interpolation_weights):
    
    mu = smooth_params(u, cluster_u, 
                       cluster_assignments=cluster_assignments, smoothing_strength=smoothing_strength, crnt_user=crnt_user, 
                       crnt_coarse_bin=crnt_coarse_bin, crnt_fine_bin_within_coarse_pos=crnt_fine_bin_within_coarse_pos, interpolation_weights=interpolation_weights)
    
    sigma2 = smooth_params(v, cluster_v, 
                        cluster_assignments=cluster_assignments, smoothing_strength=smoothing_strength, crnt_user=crnt_user, 
                        crnt_coarse_bin=crnt_coarse_bin, crnt_fine_bin_within_coarse_pos=crnt_fine_bin_within_coarse_pos, interpolation_weights=interpolation_weights)

    return mu, sigma2

In [20]:
# Creating a function that collects grid updates and a function that updates the grid

@njit
def update_grid(u, v, crnt_user_id, usr_updt_u_sum, usr_updt_v_sum, fine_bins_per_coarse_bin):
    ''' 
    Replaces the grid values using the temporary grid as data comes in
    '''
    u[crnt_user_id, : ] = usr_updt_u_sum/fine_bins_per_coarse_bin
    v[crnt_user_id, : ] = usr_updt_v_sum/fine_bins_per_coarse_bin


@njit
def collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, crnt_coarse_bin, x, mu_t, sigma_2_t, w, mean_min, var_min):
    '''
    As data comes in we update the interpolated mu values by combining with incoming data as per lambert and liu formula
    returns nothing as we modify in place
    '''
    mu_new = (1-w)*mu_t + w*x
    sigma2_new = (1-w)*sigma_2_t + w*(x-mu_t)*(x-mu_new)

    mu_new = max(mu_new, mean_min)
    sigma2_new = max(sigma2_new, var_min)


    mu_new = max(mu_new, mean_min)
    sigma2_new = max(sigma2_new, var_min)

    usr_updt_u_sum[crnt_coarse_bin] += mu_new
    usr_updt_v_sum[crnt_coarse_bin] += sigma2_new
    

In [21]:
# Function for updating cluster parameters 
@njit 
def get_new_clustering_means(cluster_groups, u, v):
    ''' 
    Calculates mean u and v values for each cluster group and each time bin
    Args:
        cluster_groups a n_users length vector of cluster assignments
        u : the calculated vector of u parameter means
        v : the calculated vector of v parameter variances
    '''

    n_users, n_coarse_bins = u.shape 
    n_clusters = cluster_groups.max() + 1

    # Init mean vectors
    cluster_mean_u = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)
    cluster_mean_v = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)
    users_per_cluster = np.zeros(n_clusters, dtype=np.float64)

    # Summing u and v contributions in each cluster
    # Extract the cluster assignment for each user and then add their parameters to each bin
    for user_id in range(n_users):
        cluster_assignment = int(cluster_groups[user_id])
        cluster_mean_u[cluster_assignment, :] += u[user_id, :]
        cluster_mean_v[cluster_assignment, :] += v[user_id, :]
        users_per_cluster[cluster_assignment] += 1

    # Dividing through to get the averages in each cluster
    for cluster_assignment in range(n_clusters):
        if users_per_cluster[cluster_assignment] > 0:
            cluster_mean_u[cluster_assignment, :] /= users_per_cluster[cluster_assignment]
            cluster_mean_v[cluster_assignment, :] /= users_per_cluster[cluster_assignment]

    return cluster_mean_u, cluster_mean_v

In [24]:
@njit
def run_lambert_liu(u_init, v_init, cluster_u_init, cluster_v_init, cluster_groups, smoothing_strength, 
                    cnts_tbl_f_bn_id, cnts_tbl_cnt, intract_tbl_frst_int, intract_tbl_lst_int, 
                    interpolation_weights, burn_in_start, test_end, fine_bins_per_week, fine_bins_per_coarse_bin, 
                    mean_min, var_min, min_mean_var_diff, w):
    ''' 
    Runs the lambert liu algorithm
    Arguents:
        u_init + v_init : inital parameter grids
        cluster_u_init + cluster_v_init : inital cluster parameters
        cluster groups : inital cluster assignments 1 row per user id
        smoothing_strength : parameter (experiment to vary)
        cnts_tbl_ ... : columns from counts_df converted to np arrays
        interact_tbl ... : columns from the interactions_table converted to np arrays
        interpolation_weights : precalculated weights for parameter interpolation
        other params come from dicts
        w : the EWMA weight for updating mu and variance as we go (as specified by lambert liu style updates)
    '''
    ###
    # Initialising grids where we will keep track of parameters and cluster mean parameters
    u = u_init.copy()
    v = v_init.copy()

    cluster_u = cluster_u_init.copy()
    cluster_v = cluster_v_init.copy()

    # Extracting bin and user numbers
    n_users, n_coarse_bins = u.shape

    # Creating a dict for output 1 row for validation and 1 row for test
    outputs = np.zeros((2,4))

    # Getting the weeks to iterate over in the data
    burn_in_first_week = burn_in_start // fine_bins_per_week
    test_last_week = test_end // fine_bins_per_week
    ###

    ##
    # Initialising a numpy array with 1 row per user which contains an index
    # the index points to the users first burn in row of counts df
    usr_frst_rw = intract_tbl_frst_int.copy()

    for user_id in range(n_users):
        cnt_tbl_idx = intract_tbl_frst_int[user_id]
        usr_lst_idx = intract_tbl_lst_int[user_id]
        
        while cnt_tbl_idx < usr_lst_idx and cnts_tbl_f_bn_id[cnt_tbl_idx] < burn_in_first_fine_bin:
            cnt_tbl_idx +=1
        usr_frst_rw[user_id] = cnt_tbl_idx
    ##

    for week in range(burn_in_first_week, test_last_week + 1):
        
        week_start = week * fine_bins_per_week
        week_end = (week + 1) * fine_bins_per_week

        if week_end > test_end:
            week_end = test_end

        # For each week iterate over the users and init the pointers
        for user_id in range(n_users):

            cnt_tbl_idx = usr_frst_rw[user_id]
            usr_end_idx = intract_tbl_lst_int[user_id]

            # Init numpy vectors for calculating the user sums
            usr_updt_u_sum = np.zeros(n_coarse_bins, dtype=np.float64)
            usr_updt_v_sum = np.zeros(n_coarse_bins, dtype=np.float64)

            for fine_bin_idx in range(week_start, week_end):
                # If we have a count in this bin make it x else make it 0
                # Move our current able index pointer
                if cnt_tbl_idx < usr_end_idx and cnts_tbl_f_bn_id[cnt_tbl_idx] == fine_bin_idx:
                    x = cnts_tbl_cnt[cnt_tbl_idx]
                    cnt_tbl_idx += 1
                else:
                    x = 0

                # Getting bin metrixs
                fine_bin_pos_in_week = fine_bin_idx % fine_bins_per_week
                cnt_coarse_bin = fine_bin_pos_in_week // fine_bins_per_coarse_bin
                crnt_fine_bin_within_coarse_pos = fine_bin_pos_in_week % fine_bins_per_coarse_bin

                # Getting the smoothed params and capping them at the minimal value
                mu_t, sigma2_t = get_smoothed_params(u, v, cluster_u, cluster_v, cluster_groups, smoothing_strength, user_id, cnt_coarse_bin, 
                                                     crnt_fine_bin_within_coarse_pos, interpolation_weights)
                mu_t = max(mu_t, mean_min)
                sigma2_t = max(sigma2_t, var_min)

                # Getting the LPMF of the observed counts
                lpmf = get_lmpf_val(x, mu_t, sigma2_t, mean_min, var_min, min_mean_var_diff)

                # Updating validation and test metrics
                time_period_int = get_time_period()
                if time_period_int in [0, 1]:
                    outputs[time_period_int, 1] += 1
                    outputs[time_period_int, 2] += lpmf
                    outputs[time_period_int, 3] += x
                    outputs[time_period_int, 4] += mu_t

                collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, usr_updt_n, cnt_coarse_bin, x, mu_t, sigma2_t, w, mean_min, var_min)
            
            # Updating the users first row (for the next week)
            usr_frst_rw[user_id] = cnt_tbl_idx

            update_grid(u, v, user_id, usr_updt_u_sum, usr_updt_v_sum, usr_updt_n)
            cluster_u, cluster_v = get_new_clustering_means(cluster_groups, u, v)

    return outputs, u, v, cluster_u, cluster_v

In [ ]:
# running the lambert liu runner

outputs, u_final, v_final, cluster_u_final, cluster_v_final = run_lambert_liu(
    u_init=u_init,
    v_init=v_init,
    cluster_u_init=clustering_model['cluster_mean_u'],
    cluster_v_init=clustering_model['cluster_mean_u'],
    cluster_groups=clustering_model['cluster_assignments'],
    smoothing_strength=config_dict["smoothing_strength"],
    cnts_tbl_f_bn_id=cnts_tbl_f_bn_id,
    cnts_tbl_cnt=cnts_tbl_cnt,
    intract_tbl_frst_int=intract_tbl_frst_int,
    intract_tbl_lst_int=intract_tbl_lst_int,
    interpolation_weights=interpolation_weights,
    burn_in_start=train_test_dict["burn_in_start"],
    validation_start=train_test_dict["validation_start"],
    validation_end=train_test_dict["validation_end"],
    test_start=train_test_dict["test_start"],
    test_end=train_test_dict["test_end"],
    fine_bins_per_week=bin_metric_dict["fine_bins_per_week"],
    fine_bins_per_coarse_bin=bin_metric_dict["fine_bins_per_coarse_bin"],
    mean_min=config_dict["mean_min"],
    var_min=config_dict["var_min"],
    min_mean_var_diff=config_dict["min_mean_var_diff"],
    w=config_dict["w"],
)

### Now unused functions

In [ ]:
def interpolate_values(values, coarse_bin_id, fine_bin_within_coarse_pos, weights):
    '''
    Takes either a set of values either U or V for each course bin and produces an 
    interpolated U/V value for the specified fine bin

    Args:
        coarse bin id : the index of the coarse bin we are currently in
        fine_bin_within_coarse_pos : the position of the fine bin within the coarse bin we are currently in
    '''
    
    # Getting the indicies of the left current and right bins to use
    n_coarse_bins = values.shape[0]
    left_bin_id = (coarse_bin_id -1) % n_coarse_bins
    right_bin_id = (coarse_bin_id + 1)% n_coarse_bins
    

    # Extract the weights we will use for the interpolation
    w_neg_1, w_0, w_1 = weights[fine_bin_within_coarse_pos]
    output = w_neg_1 * values[left_bin_id] + w_0 * values[coarse_bin_id] + w_1 * values[right_bin_id]

    return output